In [1]:
# ============================================================
# Tracker 02.09 — Customer Behavior Distributions
# Step 1: Assign Approved Customer Segments
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. SETUP
# ------------------------------------------------------------

np.random.seed(42)

project_path = Path.cwd().parent
data_path = project_path / "data" / "synthetic"

customers_file = data_path / "customers_master.csv"

print("Project path:", project_path)
print("Customer file:", customers_file)

# ------------------------------------------------------------
# 2. LOAD CUSTOMER DATA
# ------------------------------------------------------------

customers_df = pd.read_csv(customers_file)

print("\nCustomers loaded:", len(customers_df))

# ------------------------------------------------------------
# 3. REMOVE DUPLICATE CUSTOMER IDs
# ------------------------------------------------------------

customers_df = (
    customers_df
    .drop_duplicates("customer_id")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. VERIFY CUSTOMER COUNT
# ------------------------------------------------------------

if len(customers_df) != 10000:
    raise ValueError(
        f"Expected 10,000 customers, found {len(customers_df)}"
    )

# ------------------------------------------------------------
# 5. APPROVED CUSTOMER SEGMENT DISTRIBUTION
# ------------------------------------------------------------

segment_counts = {
    "Occasional": 3800,
    "Regular": 4000,
    "Premium": 1700,
    "VIP": 500
}

# ------------------------------------------------------------
# 6. CREATE SEGMENT ARRAY
# ------------------------------------------------------------

segments = []

for segment, count in segment_counts.items():
    segments.extend([segment] * count)

# Shuffle so segments are not grouped by customer ID
np.random.shuffle(segments)

# Assign segments
customers_df["customer_segment"] = segments

# ------------------------------------------------------------
# 7. APPROVED PURCHASE WEIGHTS
# ------------------------------------------------------------

purchase_weights = {
    "Occasional": 0.45,
    "Regular": 1.00,
    "Premium": 2.25,
    "VIP": 4.75
}

customers_df["purchase_frequency_weight"] = (
    customers_df["customer_segment"]
    .map(purchase_weights)
)

# ------------------------------------------------------------
# 8. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("02.09 CUSTOMER SEGMENT VALIDATION")
print("=" * 60)

segment_validation = (
    customers_df
    .groupby("customer_segment")
    .agg(
        customer_count=("customer_id", "count"),
        purchase_weight=("purchase_frequency_weight", "first")
    )
    .reset_index()
)

segment_validation["percentage"] = (
    segment_validation["customer_count"]
    / len(customers_df)
    * 100
)

print(segment_validation)

# ------------------------------------------------------------
# 9. CHECK EXPECTED COUNTS
# ------------------------------------------------------------

expected_counts = pd.Series(segment_counts)

actual_counts = (
    customers_df["customer_segment"]
    .value_counts()
    .sort_index()
)

actual_counts = actual_counts.reindex(
    expected_counts.index
)

if actual_counts.equals(expected_counts):
    print("\nSegment counts: PASSED")
else:
    print("\nSegment counts: FAILED")

# ------------------------------------------------------------
# 10. SAVE UPDATED CUSTOMER MASTER
# ------------------------------------------------------------

customers_df.to_csv(
    customers_file,
    index=False
)

print("\nUpdated customer master saved.")

print(
    "\nColumns added:"
)

print(
    "- customer_segment"
)

print(
    "- purchase_frequency_weight"
)

print(
    "\n02.09 Step 1 completed."
)

Project path: c:\Users\HP\OneDrive\Desktop\retail_sales_analytics
Customer file: c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\customers_master.csv

Customers loaded: 10000

02.09 CUSTOMER SEGMENT VALIDATION
  customer_segment  customer_count  purchase_weight  percentage
0       Occasional            3800             0.45        38.0
1          Premium            1700             2.25        17.0
2          Regular            4000             1.00        40.0
3              VIP             500             4.75         5.0

Segment counts: PASSED

Updated customer master saved.

Columns added:
- customer_segment
- purchase_frequency_weight

02.09 Step 1 completed.


In [3]:
# ============================================================
# 02.09 STEP 2 — CUSTOMER VALUE BEHAVIOR
# ============================================================

value_weights = {
    "Occasional": 0.60,
    "Regular": 1.00,
    "Premium": 1.80,
    "VIP": 3.00
}

customers_df["purchase_value_weight"] = (
    customers_df["customer_segment"]
    .map(value_weights)
)

# Validate that every customer received a value weight
missing_value_weights = customers_df["purchase_value_weight"].isna().sum()

print("Customer value behavior:")
print(
    customers_df[
        [
            "customer_segment",
            "purchase_frequency_weight",
            "purchase_value_weight"
        ]
    ]
    .drop_duplicates()
    .sort_values("purchase_value_weight")
    .to_string(index=False)
)

print()
print("Customers loaded:", len(customers_df))
print("Missing value weights:", missing_value_weights)

if missing_value_weights == 0:
    print("02.09 Step 2 validation: PASSED")
else:
    print("02.09 Step 2 validation: FAILED")

Customer value behavior:
customer_segment  purchase_frequency_weight  purchase_value_weight
      Occasional                       0.45                    0.6
         Regular                       1.00                    1.0
         Premium                       2.25                    1.8
             VIP                       4.75                    3.0

Customers loaded: 10000
Missing value weights: 0
02.09 Step 2 validation: PASSED


In [1]:
# ============================================================
# 02.09 STEP 3 — INSPECT EXISTING SALES DATA
# ============================================================

from pathlib import Path
import pandas as pd

sales_folder = Path("../data/synthetic")

sales_files = sorted(sales_folder.glob("sales_*.csv"))

print("Sales files found:", len(sales_files))

if len(sales_files) == 0:
    print("No sales files found.")
else:
    sample_file = sales_files[0]

    print("Sample file:", sample_file.name)

    sales_sample = pd.read_csv(sample_file, nrows=5)

    print("\nSales columns:")
    print(sales_sample.columns.tolist())

    print("\nSample records:")
    print(sales_sample.to_string(index=False))

    print("\nCustomer ID column check:")

    if "customer_id" in sales_sample.columns:
        print("customer_id: FOUND")
    else:
        print("customer_id: NOT FOUND")
        

Sales files found: 32
Sample file: sales_2024-01.csv

Sales columns:
['sales_line_id', 'order_id', 'order_date', 'order_time', 'customer_id', 'product_id', 'store_id', 'salesperson_id', 'promotion_id', 'sales_channel', 'quantity', 'unit_selling_price', 'discount_pct', 'payment_method', 'order_status']

Sample records:
     sales_line_id           order_id order_date      order_time customer_id product_id   store_id salesperson_id  promotion_id sales_channel  quantity  unit_selling_price  discount_pct payment_method order_status
SL2024010000000001 ORD202401000000001 2024-01-07 0 days 08:36:01  CUST008462 PROD000970 STORE00029        SP00182           NaN         Store         1            11583.24           0.0    Credit Card    Completed
SL2024010000000002 ORD202401000000002 2024-01-20 0 days 09:24:04  CUST007598 PROD000989 STORE00031        SP00256           NaN         Store         3            55589.49           0.0         Wallet    Completed
SL2024010000000003 ORD202401000000003 

In [2]:
# ============================================================
# 02.09 STEP 4 — IDENTIFY SALES BEHAVIOR COLUMNS
# ============================================================

print("Complete sales column list:")

for i, column in enumerate(sales_sample.columns, start=1):
    print(f"{i}. {column}")

print("\nSales data types:")
print(sales_sample.dtypes.to_string())

Complete sales column list:
1. sales_line_id
2. order_id
3. order_date
4. order_time
5. customer_id
6. product_id
7. store_id
8. salesperson_id
9. promotion_id
10. sales_channel
11. quantity
12. unit_selling_price
13. discount_pct
14. payment_method
15. order_status

Sales data types:
sales_line_id             str
order_id                  str
order_date                str
order_time                str
customer_id               str
product_id                str
store_id                  str
salesperson_id            str
promotion_id          float64
sales_channel             str
quantity                int64
unit_selling_price    float64
discount_pct          float64
payment_method            str
order_status              str


In [4]:
# ============================================================
# 02.09 STEP 5 — BASELINE CUSTOMER BEHAVIOR
# ============================================================

from pathlib import Path
import pandas as pd

# Project folders
sales_folder = Path("../data/synthetic")
customers_file = sales_folder / "customers_master.csv"

# Load customer master
customers_df = pd.read_csv(customers_file)

# Find sales files
sales_files = sorted(sales_folder.glob("sales_*.csv"))

print("Customers loaded:", len(customers_df))
print("Sales files found:", len(sales_files))

# Customer segment lookup
segment_lookup = (
    customers_df[
        ["customer_id", "customer_segment"]
    ]
    .drop_duplicates("customer_id")
)

customer_stats = []

# Read all monthly sales files
for file in sales_files:

    sales = pd.read_csv(file)

    # Keep transactions linked to a customer
    sales = sales[sales["customer_id"].notna()].copy()

    # Calculate sales value after discount
    sales["sales_value"] = (
        sales["quantity"]
        * sales["unit_selling_price"]
        * (1 - sales["discount_pct"] / 100)
    )

    # Customer-level monthly summary
    monthly_stats = (
        sales.groupby("customer_id")
        .agg(
            order_count=("order_id", "nunique"),
            quantity=("quantity", "sum"),
            sales_value=("sales_value", "sum")
        )
        .reset_index()
    )

    customer_stats.append(monthly_stats)

# Combine all months
customer_stats = pd.concat(
    customer_stats,
    ignore_index=True
)

# Aggregate customer behavior
customer_behavior = (
    customer_stats
    .groupby("customer_id")
    .agg(
        total_orders=("order_count", "sum"),
        total_quantity=("quantity", "sum"),
        total_sales_value=("sales_value", "sum")
    )
    .reset_index()
)

# Add customer segment
customer_behavior = customer_behavior.merge(
    segment_lookup,
    on="customer_id",
    how="left"
)

# Segment-level baseline
segment_baseline = (
    customer_behavior
    .groupby("customer_segment")
    .agg(
        customers=("customer_id", "nunique"),
        total_orders=("total_orders", "sum"),
        average_orders_per_customer=("total_orders", "mean"),
        total_quantity=("total_quantity", "sum"),
        average_quantity_per_customer=("total_quantity", "mean"),
        total_sales_value=("total_sales_value", "sum"),
        average_sales_value_per_customer=("total_sales_value", "mean")
    )
    .reset_index()
)

print("\n============================================================")
print("02.09 STEP 5 — BASELINE CUSTOMER BEHAVIOR")
print("============================================================")

print("\nSegment-level behavior:")
print(segment_baseline.to_string(index=False))

print("\nCustomer records analyzed:", len(customer_behavior))

print(
    "Customers without segment:",
    customer_behavior["customer_segment"].isna().sum()
)

print("\n02.09 Step 5 completed — baseline generated.")

Customers loaded: 10000
Sales files found: 32

02.09 STEP 5 — BASELINE CUSTOMER BEHAVIOR

Segment-level behavior:
customer_segment  customers  total_orders  average_orders_per_customer  total_quantity  average_quantity_per_customer  total_sales_value  average_sales_value_per_customer
      Occasional       3755        785000                   209.054594         1254984                     334.216778       3.823054e+10                      1.018123e+07
         Premium       1673        349134                   208.687388          559138                     334.212791       1.702285e+10                      1.017505e+07
         Regular       3948        817965                   207.184650         1308618                     331.463526       3.984527e+10                      1.009252e+07
             VIP        494        102495                   207.479757          163844                     331.668016       4.990697e+09                      1.010263e+07

Customer records analyzed: 987

In [6]:
# ============================================================
# 02.09 STEP 6 — APPLY CUSTOMER PURCHASE-FREQUENCY BEHAVIOR
# FAST VECTORIZED VERSION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

data_path = Path("../data/synthetic")

sales_files = sorted(
    data_path.glob("sales_*.csv")
)

behavior_output_path = (
    data_path / "customer_behavior_adjusted"
)

behavior_output_path.mkdir(
    parents=True,
    exist_ok=True
)

# Remove any partially generated adjusted files
for old_file in behavior_output_path.glob("sales_*.csv"):
    old_file.unlink()

# ------------------------------------------------------------
# 2. LOAD CUSTOMER MASTER
# ------------------------------------------------------------

customers_df = pd.read_csv(
    data_path / "customers_master.csv"
)

customers_df["registration_date"] = pd.to_datetime(
    customers_df["registration_date"]
)

customers_df = (
    customers_df
    .drop_duplicates("customer_id")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. APPROVED CUSTOMER BEHAVIOR WEIGHTS
# ------------------------------------------------------------

frequency_weights = {
    "Occasional": 0.45,
    "Regular": 1.00,
    "Premium": 2.25,
    "VIP": 4.75
}

customers_df["behavior_weight"] = (
    customers_df["customer_segment"]
    .map(frequency_weights)
)

if customers_df["behavior_weight"].isna().any():
    raise ValueError(
        "Customer behavior weight missing for one or more customers."
    )

# ------------------------------------------------------------
# 4. PREPARE SEGMENT ARRAYS
# ------------------------------------------------------------

segments = [
    "Occasional",
    "Regular",
    "Premium",
    "VIP"
]

segment_data = {}

for segment in segments:

    segment_df = (
        customers_df[
            customers_df["customer_segment"] == segment
        ]
        .sort_values("registration_date")
        .reset_index(drop=True)
    )

    segment_data[segment] = {
        "ids": segment_df["customer_id"].to_numpy(),
        "dates": segment_df["registration_date"].to_numpy()
    }

print("Customer behavior groups prepared.")

for segment in segments:
    print(
        segment,
        "customers:",
        len(segment_data[segment]["ids"]),
        "weight:",
        frequency_weights[segment]
    )

# ------------------------------------------------------------
# 5. PROCESS SALES FILES
# ------------------------------------------------------------

adjusted_files = []

total_rows_processed = 0
total_customer_assignments = 0
total_anonymous_rows = 0

for file_path in sales_files:

    print("\n" + "=" * 65)
    print("Processing:", file_path.name)

    sales_df = pd.read_csv(file_path)

    original_line_ids = (
        sales_df["sales_line_id"].copy()
    )

    sales_df["order_date"] = pd.to_datetime(
        sales_df["order_date"]
    )

    row_count = len(sales_df)

    # --------------------------------------------------------
    # Existing anonymous store transactions stay anonymous
    # --------------------------------------------------------

    anonymous_mask = (
        sales_df["customer_id"].isna()
        &
        (sales_df["sales_channel"] == "Store")
    )

    assign_mask = ~anonymous_mask

    assign_positions = np.flatnonzero(
        assign_mask.to_numpy()
    )

    if len(assign_positions) > 0:

        order_dates = (
            sales_df.loc[
                assign_mask,
                "order_date"
            ]
            .to_numpy()
        )

        # ----------------------------------------------------
        # Determine eligible customer count by segment
        # ----------------------------------------------------

        eligible_counts = {}

        weighted_segment_counts = []

        for segment in segments:

            segment_dates = (
                segment_data[segment]["dates"]
            )

            counts = np.searchsorted(
                segment_dates,
                order_dates,
                side="right"
            )

            eligible_counts[segment] = counts

            weighted_segment_counts.append(
                counts
                * frequency_weights[segment]
            )

        weighted_segment_counts = np.vstack(
            weighted_segment_counts
        )

        total_weights = (
            weighted_segment_counts.sum(axis=0)
        )

        # ----------------------------------------------------
        # Select customer segment
        # ----------------------------------------------------

        random_values = (
            np.random.random(len(order_dates))
            * total_weights
        )

        cumulative_weights = np.cumsum(
            weighted_segment_counts,
            axis=0
        )

        selected_segment_index = (
            random_values[None, :]
            <= cumulative_weights
        ).argmax(axis=0)

        selected_customer_ids = np.empty(
            len(order_dates),
            dtype=object
        )

        # ----------------------------------------------------
        # Select customer inside selected segment
        # ----------------------------------------------------

        for segment_index, segment in enumerate(segments):

            segment_mask = (
                selected_segment_index
                == segment_index
            )

            if not segment_mask.any():
                continue

            positions = np.flatnonzero(
                segment_mask
            )

            counts = (
                eligible_counts[segment][positions]
            )

            random_positions = (
                np.random.random(len(positions))
                * counts
            ).astype(int)

            segment_ids = (
                segment_data[segment]["ids"]
            )

            selected_customer_ids[
                positions
            ] = segment_ids[
                random_positions
            ]

        # Assign selected customers back
        sales_df.loc[
            assign_mask,
            "customer_id"
        ] = selected_customer_ids

    # --------------------------------------------------------
    # Preserve original anonymous transactions
    # --------------------------------------------------------

    sales_df.loc[
        anonymous_mask,
        "customer_id"
    ] = None

    # --------------------------------------------------------
    # Validate sales_line_id preservation
    # --------------------------------------------------------

    if not sales_df[
        "sales_line_id"
    ].equals(original_line_ids):

        raise ValueError(
            f"sales_line_id changed in {file_path.name}"
        )

    # --------------------------------------------------------
    # Save adjusted file
    # --------------------------------------------------------

    sales_df["order_date"] = (
        sales_df["order_date"]
        .dt.strftime("%Y-%m-%d")
    )

    output_file = (
        behavior_output_path
        / file_path.name
    )

    sales_df.to_csv(
        output_file,
        index=False
    )

    adjusted_files.append(
        output_file
    )

    assigned_count = (
        sales_df.loc[
            assign_mask,
            "customer_id"
        ]
        .notna()
        .sum()
    )

    anonymous_count = (
        sales_df["customer_id"]
        .isna()
        .sum()
    )

    total_rows_processed += row_count
    total_customer_assignments += assigned_count
    total_anonymous_rows += anonymous_count

    print("Rows processed:", row_count)
    print("Customer assignments:", assigned_count)
    print("Anonymous transactions:", anonymous_count)
    print("Saved:", output_file.name)

# ============================================================
# 6. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 65)
print("02.09 STEP 6 — CUSTOMER BEHAVIOR APPLICATION")
print("=" * 65)

print("Original sales files:", len(sales_files))
print("Adjusted sales files:", len(adjusted_files))
print("Total rows processed:", total_rows_processed)
print("Customer assignments:", total_customer_assignments)
print("Anonymous transactions:", total_anonymous_rows)

# ------------------------------------------------------------
# Compare total row counts
# ------------------------------------------------------------

original_total = 0
adjusted_total = 0

for file_path in sales_files:
    original_total += len(
        pd.read_csv(file_path)
    )

for file_path in adjusted_files:
    adjusted_total += len(
        pd.read_csv(file_path)
    )

print("Original total rows:", original_total)
print("Adjusted total rows:", adjusted_total)

if original_total == adjusted_total:
    print("Total row preservation: PASSED")
else:
    print("Total row preservation: FAILED")

# ------------------------------------------------------------
# Compare sales_line_id values
# ------------------------------------------------------------

original_ids = set()
adjusted_ids = set()

for file_path in sales_files:

    df = pd.read_csv(file_path)

    original_ids.update(
        df["sales_line_id"]
    )

for file_path in adjusted_files:

    df = pd.read_csv(file_path)

    adjusted_ids.update(
        df["sales_line_id"]
    )

print(
    "Original sales_line_id count:",
    len(original_ids)
)

print(
    "Adjusted sales_line_id count:",
    len(adjusted_ids)
)

if original_ids == adjusted_ids:
    print("sales_line_id preservation: PASSED")
else:
    print("sales_line_id preservation: FAILED")

print(
    "\nAdjusted files saved to:",
    behavior_output_path
)

print("\n02.09 Step 6 completed.")

Customer behavior groups prepared.
Occasional customers: 3800 weight: 0.45
Regular customers: 4000 weight: 1.0
Premium customers: 1700 weight: 2.25
VIP customers: 500 weight: 4.75

Processing: sales_2024-01.csv
Rows processed: 53200
Customer assignments: 51593
Anonymous transactions: 1607
Saved: sales_2024-01.csv

Processing: sales_2024-02.csv
Rows processed: 50400
Customer assignments: 48960
Anonymous transactions: 1440
Saved: sales_2024-02.csv

Processing: sales_2024-03.csv
Rows processed: 56000
Customer assignments: 54343
Anonymous transactions: 1657
Saved: sales_2024-03.csv

Processing: sales_2024-04.csv
Rows processed: 57100
Customer assignments: 55436
Anonymous transactions: 1664
Saved: sales_2024-04.csv

Processing: sales_2024-05.csv
Rows processed: 58800
Customer assignments: 57006
Anonymous transactions: 1794
Saved: sales_2024-05.csv

Processing: sales_2024-06.csv
Rows processed: 60500
Customer assignments: 58643
Anonymous transactions: 1857
Saved: sales_2024-06.csv

Processin

In [7]:
# ============================================================
# 02.09 STEP 6 — FINAL PRESERVATION VALIDATION
# ============================================================

from pathlib import Path
import pandas as pd

data_path = Path("../data/synthetic")

original_folder = data_path
adjusted_folder = data_path / "customer_behavior_adjusted"

original_files = sorted(
    original_folder.glob("sales_*.csv")
)

adjusted_files = sorted(
    adjusted_folder.glob("sales_*.csv")
)

print("=" * 65)
print("02.09 STEP 6 — FINAL VALIDATION")
print("=" * 65)

print("Original sales files:", len(original_files))
print("Adjusted sales files:", len(adjusted_files))

# ------------------------------------------------------------
# Row-count validation
# ------------------------------------------------------------

original_total_rows = 0
adjusted_total_rows = 0

for file_path in original_files:
    original_total_rows += len(
        pd.read_csv(file_path)
    )

for file_path in adjusted_files:
    adjusted_total_rows += len(
        pd.read_csv(file_path)
    )

print("\nOriginal total rows:", original_total_rows)
print("Adjusted total rows:", adjusted_total_rows)

if original_total_rows == adjusted_total_rows:
    print("Total row preservation: PASSED")
else:
    print("Total row preservation: FAILED")

# ------------------------------------------------------------
# Sales-line ID validation
# ------------------------------------------------------------

original_ids = set()
adjusted_ids = set()

for file_path in original_files:

    df = pd.read_csv(file_path)

    original_ids.update(
        df["sales_line_id"]
    )

for file_path in adjusted_files:

    df = pd.read_csv(file_path)

    adjusted_ids.update(
        df["sales_line_id"]
    )

print("\nOriginal sales_line_id count:", len(original_ids))
print("Adjusted sales_line_id count:", len(adjusted_ids))

if original_ids == adjusted_ids:
    print("sales_line_id preservation: PASSED")
else:
    print("sales_line_id preservation: FAILED")

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

if (
    len(original_files) == len(adjusted_files)
    and original_total_rows == adjusted_total_rows
    and original_ids == adjusted_ids
):
    print("\n02.09 Step 6 validation: PASSED")
else:
    print("\n02.09 Step 6 validation: FAILED")
    

02.09 STEP 6 — FINAL VALIDATION
Original sales files: 32
Adjusted sales files: 32

Original total rows: 2118200
Adjusted total rows: 2118200
Total row preservation: PASSED

Original sales_line_id count: 2118200
Adjusted sales_line_id count: 2118200
sales_line_id preservation: PASSED

02.09 Step 6 validation: PASSED


In [8]:
# ============================================================
# 02.09 STEP 7 — VALIDATE CUSTOMER BEHAVIOR
# ============================================================

from pathlib import Path
import pandas as pd

data_path = Path("../data/synthetic")

customers_file = data_path / "customers_master.csv"
adjusted_folder = data_path / "customer_behavior_adjusted"

# ------------------------------------------------------------
# 1. Load customer segments
# ------------------------------------------------------------

customers_df = pd.read_csv(customers_file)

segment_lookup = (
    customers_df[
        ["customer_id", "customer_segment"]
    ]
    .drop_duplicates("customer_id")
)

# ------------------------------------------------------------
# 2. Read adjusted sales files
# ------------------------------------------------------------

adjusted_files = sorted(
    adjusted_folder.glob("sales_*.csv")
)

customer_results = []

for file_path in adjusted_files:

    sales = pd.read_csv(file_path)

    # Remove anonymous transactions
    sales = sales[
        sales["customer_id"].notna()
    ].copy()

    # Calculate sales value after discount
    sales["sales_value"] = (
        sales["quantity"]
        * sales["unit_selling_price"]
        * (1 - sales["discount_pct"] / 100)
    )

    monthly_customer = (
        sales.groupby("customer_id")
        .agg(
            orders=("order_id", "nunique"),
            quantity=("quantity", "sum"),
            sales_value=("sales_value", "sum")
        )
        .reset_index()
    )

    customer_results.append(
        monthly_customer
    )

# ------------------------------------------------------------
# 3. Combine all months
# ------------------------------------------------------------

customer_results = pd.concat(
    customer_results,
    ignore_index=True
)

# ------------------------------------------------------------
# 4. Aggregate customer-level behavior
# ------------------------------------------------------------

customer_behavior = (
    customer_results
    .groupby("customer_id")
    .agg(
        total_orders=("orders", "sum"),
        total_quantity=("quantity", "sum"),
        total_sales_value=("sales_value", "sum")
    )
    .reset_index()
)

# Add customer segment
customer_behavior = customer_behavior.merge(
    segment_lookup,
    on="customer_id",
    how="left"
)

# ------------------------------------------------------------
# 5. Segment-level behavior
# ------------------------------------------------------------

segment_behavior = (
    customer_behavior
    .groupby("customer_segment")
    .agg(
        customers=("customer_id", "nunique"),
        total_orders=("total_orders", "sum"),
        avg_orders_per_customer=("total_orders", "mean"),
        avg_quantity_per_customer=("total_quantity", "mean"),
        total_sales_value=("total_sales_value", "sum"),
        avg_sales_value_per_customer=(
            "total_sales_value",
            "mean"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 6. Display result
# ------------------------------------------------------------

print("=" * 75)
print("02.09 STEP 7 — CUSTOMER BEHAVIOR VALIDATION")
print("=" * 75)

print("\nSegment behavior:")
print(
    segment_behavior.to_string(index=False)
)

# ------------------------------------------------------------
# 7. Check segment ordering
# ------------------------------------------------------------

segment_order = [
    "Occasional",
    "Regular",
    "Premium",
    "VIP"
]

behavior_check = (
    segment_behavior
    .set_index("customer_segment")
    .reindex(segment_order)
)

orders = (
    behavior_check[
        "avg_orders_per_customer"
    ]
)

values = (
    behavior_check[
        "avg_sales_value_per_customer"
    ]
)

print("\nAverage orders per customer:")
print(orders)

print("\nAverage sales value per customer:")
print(values)

# ------------------------------------------------------------
# 8. Basic validation
# ------------------------------------------------------------

orders_increasing = (
    orders.diff().dropna() > 0
).all()

values_increasing = (
    values.diff().dropna() > 0
).all()

print("\nPurchase frequency increases by segment:", orders_increasing)
print("Purchase value increases by segment:", values_increasing)

if orders_increasing and values_increasing:
    print("\n02.09 Step 7 validation: PASSED")
else:
    print("\n02.09 Step 7 validation: REVIEW REQUIRED")

02.09 STEP 7 — CUSTOMER BEHAVIOR VALIDATION

Segment behavior:
customer_segment  customers  total_orders  avg_orders_per_customer  avg_quantity_per_customer  total_sales_value  avg_sales_value_per_customer
      Occasional       3732        296762                79.518221                 127.224544       1.444888e+10                  3.871619e+06
         Premium       1674        660620               394.635603                 631.397252       3.221210e+10                  1.924259e+07
         Regular       3949        688838               174.433527                 278.944796       3.352208e+10                  8.488753e+06
             VIP        496        408300               823.185484                1317.076613       1.990629e+10                  4.013365e+07

Average orders per customer:
customer_segment
Occasional     79.518221
Regular       174.433527
Premium       394.635603
VIP           823.185484
Name: avg_orders_per_customer, dtype: float64

Average sales value per cust